# ClashAI MaskablePPO Training

Notebook-first workflow for rollout-throughput checks, MaskablePPO training, fixed-seed evaluation, and GPU utilization logging.

Use the `Python (ClashAI .venv)` kernel. From the repo root, register it with:

```bash
.venv/bin/python -m ipykernel install --user --name clashai-venv --display-name "Python (ClashAI .venv)"
```

In [7]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Keep BLAS/PyTorch CPU thread pools from fighting subprocess env workers.
for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(name, "1")

import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = Path.cwd().parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo:", REPO_ROOT)
print("python:", sys.executable)


repo: /workspace/ClashAI
python: /workspace/ClashAI/.venv/bin/python


In [8]:
from scripts.train_maskable_ppo import (
    DEFAULT_EVAL_SEED_START,
    evaluate_model,
    gpu_snapshot,
    import_training_deps,
    linear_schedule,
    make_env_factory,
    make_holdout_eval_callback,
    make_throughput_callback,
    max_preset_buildings,
    parse_army_composition,
    profile_sequence,
)
from scripts.random_baseline import summarize

deps = import_training_deps()
print("torch:", deps.torch.__version__)
print("cuda available:", deps.torch.cuda.is_available())
if deps.torch.cuda.is_available():
    for i in range(deps.torch.cuda.device_count()):
        print(f"cuda:{i}", deps.torch.cuda.get_device_name(i))
print("gpu snapshot:", gpu_snapshot())

torch: 2.12.0+cu130
cuda available: True
cuda:0 NVIDIA RTX PRO 5000 Blackwell
cuda:1 NVIDIA RTX PRO 5000 Blackwell
gpu snapshot: {'gpu_count': 2.0, 'gpu_util_pct_mean': 0.0, 'gpu_util_pct_max': 0.0, 'gpu_mem_used_gb_mean': 0.1796875, 'gpu_mem_total_gb_mean': 47.7880859375}


In [19]:
# Training configuration. First run short throughput sweeps; only then start a long pilot.
TOTAL_TIMESTEPS = 2_000_000 
PROFILE = "medium"
WORKERS = 96
DEVICE = "cuda"

RUN_HOLDOUT_EVAL = True
EVAL_FREQ = 250_000
EVAL_EPISODES = 20
N_STEPS = 512
BATCH_SIZE = 4096
EVAL_PROFILE = PROFILE

######

SEED = 42
ARMY_COMPOSITION = "barbarian=40,wall_breaker=10"

N_EPOCHS = 10
GAMMA = 0.995
GAE_LAMBDA = 0.95
CLIP_RANGE = 0.2
LEARNING_RATE = 3e-4
LINEAR_LR = False
ENT_COEF = 0.01
VF_COEF = 0.5
MAX_GRAD_NORM = 0.5
DEVICE = "cuda"           # Also sweep "cpu"; small PPO policies can be faster on CPU.
TORCH_THREADS = 1
START_METHOD = "forkserver"

EVAL_SEED_START = DEFAULT_EVAL_SEED_START
STATUS_INTERVAL_SECONDS = 30.0
CHECKPOINT_FREQ = 500_000

RUN_NAME = f"{PROFILE.replace(',', '-')}_w{WORKERS}_{DEVICE}_seed{SEED}_{time.strftime('%Y%m%d_%H%M%S')}"
LOG_DIR = Path("runs/maskable_ppo")
SAVE_DIR = Path("checkpoints/maskable_ppo") / RUN_NAME
MONITOR_DIR = Path("runs/monitor") / RUN_NAME

train_profiles = profile_sequence(PROFILE)
eval_profiles = profile_sequence(EVAL_PROFILE)
army_composition = parse_army_composition(ARMY_COMPOSITION)
max_buildings = max_preset_buildings()

print("run:", RUN_NAME)
print("train profiles:", train_profiles)
print("eval profiles:", eval_profiles)
print("army:", army_composition)
print("max_buildings:", max_buildings)
print("device:", DEVICE)
print("workers:", WORKERS)
print("holdout eval:", RUN_HOLDOUT_EVAL)


run: medium_w96_cuda_seed42_20260515_205523
train profiles: ['medium']
eval profiles: ['medium']
army: {'barbarian': 40, 'wall_breaker': 10}
max_buildings: 257
device: cuda
workers: 96
holdout eval: True


In [10]:
# Optional smoke checks before a long training run.
RUN_PYTEST = False
RUN_RANDOM_THROUGHPUT = False
RANDOM_THROUGHPUT_SECONDS = 60

if RUN_PYTEST:
    !{sys.executable} -m pytest -q

if RUN_RANDOM_THROUGHPUT:
    !{sys.executable} -B -m scripts.random_baseline_parallel --profile {PROFILE} --seconds {RANDOM_THROUGHPUT_SECONDS} --workers {WORKERS}


In [20]:
_required_config = [
    "TORCH_THREADS", "WORKERS", "START_METHOD", "SAVE_DIR", "MONITOR_DIR",
    "train_profiles", "max_buildings", "army_composition", "SEED", "N_STEPS",
    "BATCH_SIZE", "N_EPOCHS", "GAMMA", "GAE_LAMBDA", "CLIP_RANGE",
    "LEARNING_RATE", "LINEAR_LR", "ENT_COEF", "VF_COEF", "MAX_GRAD_NORM",
    "LOG_DIR", "DEVICE",
]
_missing = [name for name in _required_config if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the training configuration cell before creating the env/model. Missing: {_missing}")

if TORCH_THREADS > 0:
    deps.torch.set_num_threads(TORCH_THREADS)

# Re-running this cell in a notebook should not leave old subprocess workers alive.
try:
    env.close()
except NameError:
    pass

SAVE_DIR.mkdir(parents=True, exist_ok=True)
MONITOR_DIR.mkdir(parents=True, exist_ok=True)

env_fns = []
for rank in range(WORKERS):
    profile = train_profiles[rank % len(train_profiles)]
    seed = SEED + rank * 100_003
    env_fns.append(make_env_factory(
        profile=profile,
        seed=seed,
        max_buildings=max_buildings,
        army_composition=army_composition,
        monitor_cls=deps.Monitor,
        monitor_file=MONITOR_DIR / f"worker_{rank}",
    ))

VecEnv = deps.DummyVecEnv if WORKERS == 1 else deps.SubprocVecEnv
env = VecEnv(env_fns) if WORKERS == 1 else VecEnv(env_fns, start_method=START_METHOD)
learning_rate = linear_schedule(LEARNING_RATE) if LINEAR_LR else LEARNING_RATE

model = deps.MaskablePPO(
    deps.MaskableMultiInputActorCriticPolicy,
    env,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    clip_range=CLIP_RANGE,
    learning_rate=learning_rate,
    ent_coef=ENT_COEF,
    vf_coef=VF_COEF,
    max_grad_norm=MAX_GRAD_NORM,
    tensorboard_log=str(LOG_DIR),
    device=DEVICE,
    verbose=1,
)
model


Using cuda device


In [21]:
_required_training_state = [
    "model", "WORKERS", "CHECKPOINT_FREQ", "SAVE_DIR", "RUN_HOLDOUT_EVAL",
    "EVAL_FREQ", "eval_profiles", "EVAL_SEED_START", "EVAL_EPISODES",
    "max_buildings", "army_composition", "TOTAL_TIMESTEPS", "RUN_NAME",
]
_missing = [name for name in _required_training_state if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the config and model-construction cells before training. Missing: {_missing}")

ThroughputCallback = make_throughput_callback(deps.BaseCallback)
HoldoutEvalCallback = make_holdout_eval_callback(deps.BaseCallback)

callback_items = [
    ThroughputCallback(interval_seconds=STATUS_INTERVAL_SECONDS),
    deps.CheckpointCallback(
        save_freq=max(CHECKPOINT_FREQ // WORKERS, 1),
        save_path=str(SAVE_DIR),
        name_prefix="checkpoint",
        save_replay_buffer=False,
        save_vecnormalize=False,
    ),
]

if RUN_HOLDOUT_EVAL:
    callback_items.append(HoldoutEvalCallback(
        eval_freq=EVAL_FREQ,
        profiles=eval_profiles,
        seed_start=EVAL_SEED_START,
        episodes=EVAL_EPISODES,
        max_buildings=max_buildings,
        army_composition=army_composition,
        deterministic=True,
        best_model_path=SAVE_DIR / "best_model",
    ))

callbacks = deps.CallbackList(callback_items)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callbacks,
    tb_log_name=RUN_NAME,
    log_interval=1,
    progress_bar=False,
)

final_path = SAVE_DIR / "final_model"
model.save(str(final_path))
print("saved:", final_path)


Logging to runs/maskable_ppo/medium_w96_cuda_seed42_20260515_205523_1
timesteps=9312 | env_steps/sec=307.4 | episodes/sec=3.17 | gpu_util_mean=0% gpu_util_max=1% | gpu_mem_mean=0.8/47.8GB
timesteps=17952 | env_steps/sec=286.9 | episodes/sec=3.52 | gpu_util_mean=0% gpu_util_max=1% | gpu_mem_mean=0.8/47.8GB
timesteps=27168 | env_steps/sec=305.0 | episodes/sec=5.66 | gpu_util_mean=0% gpu_util_max=1% | gpu_mem_mean=0.8/47.8GB
timesteps=37056 | env_steps/sec=325.9 | episodes/sec=4.05 | gpu_util_mean=0% gpu_util_max=1% | gpu_mem_mean=0.8/47.8GB
timesteps=46368 | env_steps/sec=308.8 | episodes/sec=5.21 | gpu_util_mean=0% gpu_util_max=1% | gpu_mem_mean=0.8/47.8GB
---------------------------------------
| gpu/                     |          |
|    gpu_count             | 2        |
|    gpu_mem_total_gb_mean | 47.8     |
|    gpu_mem_used_gb_mean  | 0.757    |
|    gpu_util_pct_max      | 1        |
|    gpu_util_pct_mean     | 0.5      |
| rollout/                 |          |
|    ep_len_mean

In [ ]:
# Fixed holdout evaluation for the current in-memory model.
results, elapsed = evaluate_model(
    model,
    profiles=eval_profiles,
    seed_start=EVAL_SEED_START,
    episodes=20,
    max_buildings=max_buildings,
    army_composition=army_composition,
    deterministic=True,
)
print(summarize(results, elapsed))


In [ ]:
# TensorBoard inside Jupyter. If this fails, run from a terminal:
# .venv/bin/tensorboard --logdir runs/maskable_ppo
%load_ext tensorboard
%tensorboard --logdir runs/maskable_ppo --host 0.0.0.0

In [ ]:
# Close subprocess workers when done with the notebook session.
env.close()